In [1]:
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import from_bounds
from rasterio.features import rasterize

fire = gpd.read_file("../data/processed/pilot/fire_2020_112.gpkg")
assert fire.crs.to_epsg() == 4326

# buffers in metres (UTM zone 10N), then back to lat/lon
utm = fire.to_crs(32610)
inner = utm.buffer(-500).to_crs(4326)      # burn minus outer 500 m
excl  = utm.buffer(1000).to_crs(4326)      # burn plus 1 km margin, excluded from controls

pad = 0.15
west, south, east, north = fire.total_bounds
west, south, east, north = west - pad, south - pad, east + pad, north + pad

def read_year(kind, yr):
    f = f"../data/cci_agb/N40W130_ESACCI-BIOMASS-L4-{kind}-MERGED-100m-{yr}-fv6.0.tif"
    with rasterio.open(f) as src:
        assert src.crs.to_epsg() == 4326
        win = from_bounds(west, south, east, north, src.transform).round_offsets().round_lengths()
        arr = src.read(1, window=win, masked=True).astype("float64").filled(np.nan)
        return arr, src.window_transform(win), src.nodata

agb, sd, tr_ref = {}, {}, None
for yr in (2019, 2021, 2022):
    agb[yr], tr, nd = read_year("AGB", yr)
    sd[yr], tr_sd, _ = read_year("AGB_SD", yr)
    tr_ref = tr_ref or tr
    assert tr == tr_ref == tr_sd, "grids differ"
print("nodata value:", nd, "| window shape:", agb[2019].shape)

shape = agb[2019].shape
m_inner = rasterize([(g, 1) for g in inner.geometry], out_shape=shape,
                    transform=tr_ref, fill=0, dtype="uint8").astype(bool)
m_excl  = rasterize([(g, 1) for g in excl.geometry], out_shape=shape,
                    transform=tr_ref, fill=0, dtype="uint8").astype(bool)

valid = (np.isfinite(agb[2019]) & np.isfinite(agb[2021]) & np.isfinite(agb[2022])
         & (agb[2019] > 10))          # forested pixels only; 10 Mg/ha is a placeholder threshold

def summary(mask, label):
    m = mask & valid
    b = agb[2019][m]
    out = {"label": label, "n_pixels": int(m.sum()), "mean_agb_2019": b.mean(),
           "mean_sd_2019": sd[2019][m].mean()}
    for yr in (2021, 2022):
        a = agb[yr][m]
        out[f"loss_frac_{yr}"] = 1 - a.sum() / b.sum()
        out[f"mean_change_{yr}"] = (a - b).mean()
    return out

res = pd.DataFrame([summary(m_inner, "burned (inner)"),
                    summary(~m_excl, "control (outside fire +1 km)")]).set_index("label").T
print(res)

m = m_inner & valid
loss = np.clip(1 - agb[2021][m] / agb[2019][m], 0, 1)
print("per-pixel loss share, 2021, percentiles (5,25,50,75,95):", np.percentile(loss, [5, 25, 50, 75, 95]))

nodata value: None | window shape: (717, 939)
label             burned (inner)  control (outside fire +1 km)
n_pixels           125856.000000                 338949.000000
mean_agb_2019          90.157068                     86.055274
mean_sd_2019           53.197392                     54.181992
loss_frac_2021          0.028862                     -0.046501
mean_change_2021       -2.602077                      4.001691
loss_frac_2022          0.059477                     -0.023915
mean_change_2022       -5.362239                      2.058050
per-pixel loss share, 2021, percentiles (5,25,50,75,95): [0.         0.         0.06590116 0.41347961 0.76190476]
